# Notebook 06 — Visualizaciones finales

Aquí regenero todas las gráficas del proyecto con un estilo unificado, además del banner para el README y la portada del carrusel de LinkedIn. Los gráficos exploratorios viven en los notebooks 02–05; este notebook produce las versiones publicables.

Salida: cinco PNGs en `outputs/` más el `banner.png`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import enoe_loader as enoe

# Paleta unificada
AZUL    = "#2c7fb8"   # hombres / horas explicadas
NARANJA = "#d95f0e"   # mujeres / residual
GRIS    = "#666666"   # texto secundario
AZUL_CLARO = "#a6bddb"

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 180, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
})

OUTPUTS = ROOT / "outputs"
df   = pd.read_parquet(ROOT / "data" / "processed" / "enoen_2025_4t_mincer.parquet")
boot = pd.read_csv(OUTPUTS / "oaxaca_bootstrap.csv")
h = df[df["mujer"]==0]; m = df[df["mujer"]==1]


## 1. Banner del proyecto

Imagen de portada para README y post de LinkedIn. Formato 12×6.3 (cerca de 1200×630 de redes).


In [ ]:
brecha = boot["brecha_log"].median()
horas_pct  = boot["aporte_horas"].median() / brecha * 100
no_exp_pct = boot["no_explicado"].median() / brecha * 100
horas_lo, horas_hi = np.percentile(boot["aporte_horas"]/boot["brecha_log"]*100, [2.5, 97.5])
no_lo, no_hi       = np.percentile(boot["no_explicado"]/boot["brecha_log"]*100, [2.5, 97.5])

fig, ax = plt.subplots(figsize=(12, 6.3))
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")

ax.text(50, 92, "La brecha salarial en México",
        ha="center", fontsize=24, fontweight="bold", color="#222")
ax.text(50, 83, "no es lo que crees", ha="center",
        fontsize=24, fontweight="bold", color="#222", style="italic")
ax.text(50, 71,
        "De la brecha mensual del 32%, casi la mitad la explican las horas trabajadas.\n"
        "La otra mitad no se explica con educación, sector ni geografía.",
        ha="center", fontsize=13, color=GRIS)

y_barra, alto, ancho_total, left = 38, 16, 80, 10
x = left
for v, color, etiqueta, ic in [
    (horas_pct,  AZUL,    "Horas trabajadas",         (horas_lo, horas_hi)),
    (no_exp_pct, NARANJA, "No explicado\n(residual)", (no_lo, no_hi)),
]:
    w = v * ancho_total / 100
    ax.add_patch(Rectangle((x, y_barra), w, alto, color=color, alpha=0.92, ec="white", lw=2))
    ax.text(x+w/2, y_barra+alto/2+1, f"{v:.0f}%", ha="center", va="center",
            color="white", fontweight="bold", fontsize=30)
    ax.text(x+w/2, y_barra+alto+4, etiqueta, ha="center", va="bottom",
            color=color, fontweight="bold", fontsize=13)
    ax.text(x+w/2, y_barra-4, f"IC 95%: [{ic[0]:.0f}%, {ic[1]:.0f}%]",
            ha="center", va="top", color=GRIS, fontsize=9.5)
    x += w

ax.text(50, 12,
        "Microdatos ENOE 4T 2025 (INEGI) · 110,563 personas ocupadas · descomposición Oaxaca-Blinder",
        ha="center", fontsize=10, color=GRIS, style="italic")
plt.savefig(OUTPUTS / "banner.png", facecolor="white")
plt.show()


## 2. Las cinco gráficas finales

Cada una con un mensaje claro en el título, anotaciones directas y créditos al pie.


### G1 — Caracterización Mincer por sexo

La imagen que rompe el cliché. Las mujeres tienen más educación, igual experiencia, más formalidad, y solo trabajan menos horas.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
axes = axes.flatten()
casos = [
    ("Escolaridad (años)",  "anios_escolaridad", 12, "+{:.1f} años a favor de mujeres", False),
    ("Experiencia (años)",  "experiencia",       27, "Prácticamente iguales",            False),
    ("% en sector formal",  "subordinado",       65, "+{:.1f} pp a favor de mujeres",   False),
    ("Horas semanales",     "horas_semanales",   52, "−{:.1f} hrs en mujeres — única diferencia grande", True),
]
for ax, (titulo, col, xmax, plantilla, es_neg) in zip(axes, casos):
    serie_h = h[col].astype(float) if col=="subordinado" else h[col]
    serie_m = m[col].astype(float) if col=="subordinado" else m[col]
    vh = enoe.media_ponderada(serie_h, h["fac_tri"])
    vm = enoe.media_ponderada(serie_m, m["fac_tri"])
    if col == "subordinado":
        vh, vm = vh*100, vm*100; sufijo = "%"
    else:
        sufijo = ""
    ax.barh(["Hombres","Mujeres"], [vh, vm], color=[AZUL, NARANJA], alpha=0.85)
    ax.set_xlim(0, xmax)
    diff = abs(vm - vh)
    color_tit = "#b32b1f" if es_neg else "black"
    ax.set_title(f"{titulo}\n{plantilla.format(diff)}", loc="left", fontsize=11, color=color_tit)
    ax.text(vh*0.97, 0, f"{vh:.1f}{sufijo}", va="center", ha="right", color="white", fontweight="bold")
    ax.text(vm*0.97, 1, f"{vm:.1f}{sufijo}", va="center", ha="right", color="white", fontweight="bold")
    ax.set_xticks([])

fig.suptitle("Las mujeres ocupadas mexicanas tienen más educación y más formalidad.\n"
             "La única variable Mincer donde están en desventaja son las horas trabajadas.",
             fontsize=12.5, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS / "final_g1_caracterizacion.png")
plt.show()


### G2 — La descomposición Oaxaca-Blinder

El gráfico que sostiene el hallazgo central.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
izq = 0
for v, c, et, (lo, hi) in [
    (horas_pct,  AZUL,    "Horas trabajadas",          (horas_lo, horas_hi)),
    (no_exp_pct, NARANJA, "No explicado\n(residual)", (no_lo, no_hi)),
]:
    ax.barh(["Brecha mensual"], [v], left=[izq], color=c, edgecolor="white", linewidth=2, height=0.5)
    ax.text(izq+v/2, 0, f"{v:.1f}%", va="center", ha="center", color="white",
            fontweight="bold", fontsize=14)
    ax.text(izq+v/2, -0.35, f"IC95: [{lo:.1f}, {hi:.1f}]",
            va="center", ha="center", color=GRIS, fontsize=8.5)
    ax.text(izq+v/2, 0.45, et, va="center", ha="center", fontsize=10.5,
            fontweight="bold", color=c)
    izq += v
ax.set_xlim(-3, 105); ax.set_xticks([]); ax.set_yticks([])
ax.spines["left"].set_visible(False); ax.spines["bottom"].set_visible(False)
ax.text(50, 1.05, "De la brecha mensual del 32%, la mitad la explican las horas trabajadas.\n"
        "La otra mitad no se explica con variables observables.",
        ha="center", fontsize=13, fontweight="bold", transform=ax.transData)
ax.text(50, -0.85, "Descomposición Oaxaca-Blinder pooled (Neumark) · IC 95% bootstrap (240 réplicas) · "
        "ENOE 4T 2025 · INEGI",
        ha="center", fontsize=8.5, color=GRIS, transform=ax.transData)
plt.tight_layout()
plt.savefig(OUTPUTS / "final_g2_descomposicion.png")
plt.show()


### G3 — Distribución de horas

Para el slide 3 del carrusel. La doble distribución femenina muestra que no es "todas trabajan menos" — es "una parte trabaja igual, la otra mitad trabaja jornada parcial".


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.arange(0, 80, 2)
ax.hist(h["horas_semanales"], bins=bins, weights=h["fac_tri"]/h["fac_tri"].sum(),
        alpha=0.55, color=AZUL, label="Hombres")
ax.hist(m["horas_semanales"], bins=bins, weights=m["fac_tri"]/m["fac_tri"].sum(),
        alpha=0.55, color=NARANJA, label="Mujeres")
hrs_h = enoe.media_ponderada(h["horas_semanales"], h["fac_tri"])
hrs_m = enoe.media_ponderada(m["horas_semanales"], m["fac_tri"])
ax.axvline(hrs_h, color=AZUL, linestyle="--", linewidth=1.5)
ax.axvline(hrs_m, color=NARANJA, linestyle="--", linewidth=1.5)
ax.annotate(f"{hrs_h:.1f} hrs", xy=(hrs_h, 0.08), xytext=(hrs_h+7, 0.085),
            color=AZUL, fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color=AZUL))
ax.annotate(f"{hrs_m:.1f} hrs", xy=(hrs_m, 0.10), xytext=(hrs_m-15, 0.105),
            color=NARANJA, fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="-", color=NARANJA))
ax.set_xlabel("Horas semanales trabajadas")
ax.set_ylabel("Proporción de personas ocupadas")
ax.set_title(f"Las mujeres trabajan {hrs_h-hrs_m:.1f} hrs menos por semana en el mercado remunerado")
ax.text(0.99, -0.18, "ENOE 4T 2025 — INEGI. Ponderado por factor de expansión.",
        transform=ax.transAxes, ha="right", fontsize=8, color=GRIS)
ax.legend(loc="upper right")
plt.savefig(OUTPUTS / "final_g3_horas.png")
plt.show()


### G4 — Distribución bootstrap de los componentes

Para mostrar la robustez estadística del hallazgo.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, col, titulo, color in zip(axes,
    ["aporte_horas", "aporte_composicion", "no_explicado"],
    ["Aporte de horas", "Aporte de composición", "No explicado (residual)"],
    [AZUL, AZUL_CLARO, NARANJA]):
    pcts = boot[col] / boot["brecha_log"] * 100
    ax.hist(pcts, bins=25, color=color, alpha=0.85, edgecolor="white")
    med, lo, hi = pcts.median(), np.percentile(pcts, 2.5), np.percentile(pcts, 97.5)
    ax.axvline(med, color="black", linewidth=1.5)
    ax.axvline(lo,  color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.axvline(hi,  color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.axvline(0,   color="red",   linewidth=0.8, alpha=0.5)
    ax.set_title(f"{titulo}\nMediana: {med:.1f}%  |  IC95: [{lo:.1f}, {hi:.1f}]", fontsize=11)
    ax.set_xlabel("% de la brecha total")
axes[0].set_ylabel("Frecuencia (240 réplicas)")
plt.suptitle("Distribución bootstrap de los componentes de la brecha",
             fontsize=12.5, fontweight="bold", y=1.04)
plt.tight_layout()
plt.savefig(OUTPUTS / "final_g4_distribuciones.png")
plt.show()


### G5 — Brecha por entidad federativa

Para el slide de geografía del carrusel.


In [ ]:
ESTADOS = {
    1:"Aguascalientes", 2:"Baja California", 3:"Baja California Sur", 4:"Campeche",
    5:"Coahuila", 6:"Colima", 7:"Chiapas", 8:"Chihuahua", 9:"Ciudad de México",
    10:"Durango", 11:"Guanajuato", 12:"Guerrero", 13:"Hidalgo", 14:"Jalisco",
    15:"Estado de México", 16:"Michoacán", 17:"Morelos", 18:"Nayarit", 19:"Nuevo León",
    20:"Oaxaca", 21:"Puebla", 22:"Querétaro", 23:"Quintana Roo", 24:"San Luis Potosí",
    25:"Sinaloa", 26:"Sonora", 27:"Tabasco", 28:"Tamaulipas", 29:"Tlaxcala",
    30:"Veracruz", 31:"Yucatán", 32:"Zacatecas",
}
filas = []
for ent_id, grp in df.groupby("ent", observed=True):
    hh = grp[grp["mujer"]==0]; mm = grp[grp["mujer"]==1]
    if len(hh) < 30 or len(mm) < 30: continue
    ing_h = enoe.media_ponderada(hh["ingreso_mensual_w"], hh["fac_tri"])
    ing_m = enoe.media_ponderada(mm["ingreso_mensual_w"], mm["fac_tri"])
    filas.append({"entidad": ESTADOS[int(ent_id)],
                   "brecha": (ing_h-ing_m)/ing_h*100})
brecha_ent = pd.DataFrame(filas).sort_values("brecha")
prom = brecha_ent["brecha"].mean()

fig, ax = plt.subplots(figsize=(10, 9))
colores = [AZUL_CLARO if v < prom else AZUL for v in brecha_ent["brecha"]]
ax.barh(brecha_ent["entidad"], brecha_ent["brecha"], color=colores, alpha=0.9)
for i, v in enumerate(brecha_ent["brecha"]):
    ax.text(v+0.4, i, f"{v:.1f}%", va="center", fontsize=8.5)
ax.axvline(prom, color=GRIS, linestyle="--", alpha=0.7)
ax.text(prom+0.3, -1.6, f"Promedio nacional: {prom:.1f}%", color=GRIS, fontsize=9)
ax.set_xlabel("Brecha mensual de ingreso (%)")
ax.set_title("La brecha mensual varía 4× entre entidades — Chiapas 8%, Hidalgo 32%")
ax.text(0.99, -0.04, "ENOE 4T 2025 — INEGI. Entidades con ≥30 registros por sexo.",
        transform=ax.transAxes, ha="right", fontsize=8, color=GRIS)
plt.savefig(OUTPUTS / "final_g5_entidades.png")
plt.show()


## Cierre del proyecto

Los entregables del portafolio están listos:

- **README.md** con narrativa completa, gráficas, debrief y reproducibilidad.
- **outputs/banner.png** como imagen del post de LinkedIn.
- **outputs/final_g1–g5*.png** como slides del carrusel.
- **post_linkedin.md** con texto del post y versión carrusel.

Lo que falta antes de publicar:
- Subir el repo a GitHub con `git init && git push`.
- Verificar que `.gitignore` excluye los CSVs crudos.
- Sustituir `darioomar-blip` en el README y banner con el handle real de GitHub.
- Publicar martes o miércoles de la primera semana de julio 2026 entre 8 y 10am hora CDMX.

Y actualizar `proyectos_realizados.md` con la entrada del proyecto 2.
